In [1]:
import sys
import os
from scripts.kml_utils import parse_kml_coordinates
import ee
import geemap
import folium
from colorama import Fore, Back, Style

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm, gamma, f, chi2
import IPython.display as disp
%matplotlib inline

#my helper here
# parse_kml_coordinates already imported above

# The main idea of this notebook is extract the embeddings
# more info about embeddings here:
# https://developers.google.com/machine-learning/crash-course/embeddings/
# https://developers.google.com/machine-learning/crash-course/embeddings/embedding-space
# something defenitivily worthy to study:
# https://deepmind.google.com/science/weatherlab

# Initialize the Google Earth Engine module in your notebook session before running the cells below.
ee.Authenticate()
# This is my project, to replicate this experiment you should go to google earth engine and copy your Project ID in the line below
ee.Initialize(project='eastern-thinker-471320-h4')

# The embeddings are visualized for two different years (2023 and 2024),
# and a dot product is calculated to measure the similarity between the
# embedding vectors of the two years.
# Create map
Map = geemap.Map()
Map.add_basemap('SATELLITE')

# Define a method for displaying Earth Engine image tiles to folium map.
def add_ee_layer(self, ee_image_object, vis_params, name):
  map_id_dict = ee.Image(ee_image_object).getMapId(vis_params)
  folium.raster_layers.TileLayer(
    tiles = map_id_dict['tile_fetcher'].url_format,
    attr = 'Map Data &copy; <a href="https://earthengine.google.com/">Google Earth Engine</a>',
    name = name,
    overlay = True,
    control = True
  ).add_to(self)

# Add EE drawing method to folium.
folium.Map.add_ee_layer = add_ee_layer


# Parse KML and build geometry
# highly dense polygon
coords = parse_kml_coordinates('BlackHills.kml')

# medium dense polygon
# coords = parse_kml_coordinates('BlackHills_reduced_smoothed.kml')

# low dense polygon
# coords = parse_kml_coordinates('BlackHills_reduced.kml')

#print(Fore.YELLOW + 'Coordinates:' + Fore.RESET, coords)

geoJSON = {
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {},
      "geometry": {
        "type": "Polygon",
        "coordinates": 
          coords
      }
    }
  ]
}
#print(geoJSON)
# Create a polygon geometry
blackHills_geom = ee.Geometry.Polygon(coords)
#print(Back.RED + 'Geometry' + Back.RESET , blackHills_geom)


# Visualization params
vis_params = {
    'min': 0,
    'max': 4000,
    'palette': ['006633', 'E5FFCC', '662A00', 'D8D8D8', 'F5F5F5']
}


# new code from above
coords = geoJSON['features'][0]['geometry']['coordinates']
aoi_blackhills = ee.Geometry.Polygon(coords)

ffa_db = ee.Image(ee.ImageCollection('COPERNICUS/S1_GRD') 
                       .filterBounds(aoi_blackhills) 
                       .filterDate(ee.Date('2020-08-01'), ee.Date('2020-08-31')) 
                       .first() 
                       .clip(aoi_blackhills))
ffa_fl = ee.Image(ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT') 
                       .filterBounds(aoi_blackhills) 
                       .filterDate(ee.Date('2020-08-01'), ee.Date('2020-08-31')) 
                       .first() 
                       .clip(aoi_blackhills))


ffa_db.bandNames().getInfo()


#start here
# Load collection
dataset = ee.Image(ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL') 
                       .filterBounds(aoi_blackhills) 
                       .filterDate(ee.Date('2020-08-01'), ee.Date('2020-08-31')) 
                       .first() 
                       .clip(aoi_blackhills))

                      
location = aoi_blackhills.centroid().coordinates().getInfo()[::-1]
print(Back.RED + 'Geometry' + Back.RESET , location)
#end here



# Make an RGB color composite image (VV,VH,VV/VH).
rgb = ee.Image.rgb(ffa_db.select('VV'),
                   ffa_db.select('VH'),
                   ffa_db.select('VV').divide(ffa_db.select('VH')))

# Create the map object.
m = folium.Map(location=location, zoom_start=7)

# Add the S1 rgb composite to the map object.
m.add_ee_layer(rgb, {'min': [-20, -20, 0], 'max': [0, 0, 2]}, 'David')

# Add a layer control panel to the map.
#m.add_child(folium.LayerControl())
# Display the map.
display(m)

/Users/david/dev/python/google-earth-engine/venv-geemap/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/david/dev/python/google-earth-engine/venv-geemap/lib/python3.9/site-packages/geemap/conversion.py:24: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Geometry [43.988020092438674, -103.73714829146317]


In [2]:
"""
# Get embedding images for two years.
# Filter for 2023 (Jan 1, 2023 to Dec 31, 2023)
image1 = dataset.filterDate('2023-01-01', '2024-01-01').filterBounds(blackHills_geom).first()
print(Back.RED + 'Image info' + Back.RESET , image1.getInfo())
print(Back.GREEN + 'Image bands' + Back.RESET , image1.bandNames().getInfo())
print(Back.GREEN + 'Image projection' + Back.RESET , image1.projection().getInfo())
print(Back.GREEN + 'Image system:time_start' + Back.RESET , image1.get('system:time_start').getInfo())
print(Back.GREEN + 'Image system:time_end' + Back.RESET , image1.get('system:time_end').getInfo())


# Filter for 2024 (Jan 1, 2024 to Dec 31, 2024)
image2 = dataset.filterDate('2024-01-01', '2025-01-01').filterBounds(blackHills_geom).first()

# Visualize three axes of the embedding space as an RGB.
vis_params = {'min': -0.3, 'max': 0.3, 'bands': ['A01', 'A16', 'A09']}
# Map centered on the Black Hills region 
lat = 44.001694
lon = -103.653595
tiles = 'http://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}'
attr = '&copy; Google'
m = folium.Map(location=[lat, lon], zoom_start=8, tiles=tiles, attr=attr)


# function to add Earth Engine layers to a Folium map
def add_ee_layer(m, ee_image, vis_params, name):
    tile_url = ee_image.getMapId(vis_params)['tile_fetcher'].url_format
    folium.TileLayer(
        tiles=tile_url,
        attr='Google Earth Engine',
        name=name,
        overlay=True,
        control=True
    ).add_to(m)


add_ee_layer(m, image1.clip(blackHills_geom.buffer(0)), vis_params, '2023 embeddings')
add_ee_layer(m, image2.clip(blackHills_geom.buffer(0)), vis_params, '2024 embeddings')


# Calculate dot product as a measure of similarity between embedding vectors.
# Note: The 'reduce' function will apply the sum over the bands (A01 to A16)
dot_prod = image1.multiply(image2).reduce(ee.Reducer.sum())

# Add dot product to the map.
dot_prod_vis = {'min': 0, 'max': 1, 'palette': ['white', 'black']}
add_ee_layer(m,dot_prod.clip(blackHills_geom.buffer(0)),dot_prod_vis,'Similarity between years (brighter = less similar)')

folium.LayerControl().add_to(m)
display(m)
"""

"\n# Get embedding images for two years.\n# Filter for 2023 (Jan 1, 2023 to Dec 31, 2023)\nimage1 = dataset.filterDate('2023-01-01', '2024-01-01').filterBounds(blackHills_geom).first()\nprint(Back.RED + 'Image info' + Back.RESET , image1.getInfo())\nprint(Back.GREEN + 'Image bands' + Back.RESET , image1.bandNames().getInfo())\nprint(Back.GREEN + 'Image projection' + Back.RESET , image1.projection().getInfo())\nprint(Back.GREEN + 'Image system:time_start' + Back.RESET , image1.get('system:time_start').getInfo())\nprint(Back.GREEN + 'Image system:time_end' + Back.RESET , image1.get('system:time_end').getInfo())\n\n\n# Filter for 2024 (Jan 1, 2024 to Dec 31, 2024)\nimage2 = dataset.filterDate('2024-01-01', '2025-01-01').filterBounds(blackHills_geom).first()\n\n# Visualize three axes of the embedding space as an RGB.\nvis_params = {'min': -0.3, 'max': 0.3, 'bands': ['A01', 'A16', 'A09']}\n# Map centered on the Black Hills region \nlat = 44.001694\nlon = -103.653595\ntiles = 'http://mt1.goo

In [3]:
"""
# now the core Earth Engine calculations

# Calculate dot product as a measure of similarity between embedding vectors.
# Note for vectors with a magnitude of 1, this simplifies to the cosine of the
# angle between embedding vectors.
dot_prod = image1.multiply(image2).reduce(ee.Reducer.sum())

# Define the visualization parameters for the dot product image
dot_prod_vis = {'min': 0, 'max': 1, 'palette': ['white', 'black']}

# Output the result
# To get the value of the dot product at the exact point, use reduceRegion:
similarity_value = dot_prod.reduceRegion(
    reducer=ee.Reducer.mean(), # Mean is appropriate for a single pixel point
    geometry=blackHills_geom.centroid(), # Use the centroid of the geometry
    scale=10 # Use the native scale of the imagery or a suitable one
).get('sum') # The band name is 'sum' because we used ee.Reducer.sum() for the dot_prod image
print(f'Similarity value at the point: {similarity_value.getInfo()}')

print("\nEarth Engine Operations completed successfully.")
print(f"The calculated similarity value (dot product) at the point is: {similarity_value.getInfo()}")
"""

'\n# now the core Earth Engine calculations\n\n# Calculate dot product as a measure of similarity between embedding vectors.\n# Note for vectors with a magnitude of 1, this simplifies to the cosine of the\n# angle between embedding vectors.\ndot_prod = image1.multiply(image2).reduce(ee.Reducer.sum())\n\n# Define the visualization parameters for the dot product image\ndot_prod_vis = {\'min\': 0, \'max\': 1, \'palette\': [\'white\', \'black\']}\n\n# Output the result\n# To get the value of the dot product at the exact point, use reduceRegion:\nsimilarity_value = dot_prod.reduceRegion(\n    reducer=ee.Reducer.mean(), # Mean is appropriate for a single pixel point\n    geometry=blackHills_geom.centroid(), # Use the centroid of the geometry\n    scale=10 # Use the native scale of the imagery or a suitable one\n).get(\'sum\') # The band name is \'sum\' because we used ee.Reducer.sum() for the dot_prod image\nprint(f\'Similarity value at the point: {similarity_value.getInfo()}\')\n\nprint("\n